# Desafio — ¿Dentro o fuera de las 12 millas?
## Peritaje de datos VMS en un sumario pesquero (caso simulado)

**Curso:** IA Aplicada a la Produccion Pesquera · UTN FRCh · PesquerosEnIA · 2026
**Clase:** 4 — Datos y sensores (VMS, cartografia oficial). Se retoma en Clase 6 (incertidumbre) y Clase 8 (asistentes de IA).
**Inspirado en:** un caso real aportado por una alumna del curso (gestoria pesquera), recreado con datos ficticios.

> 🟥 **DATOS SIMULADOS.** Este desafio imita un sumario real, pero el buque *DON FICTICIO*, el armador, las personas, el expediente, las posiciones y las cifras son **ficticios**. Solo son reales y publicas las **normas** citadas y las **capas oficiales** del Servicio de Hidrografia Naval (via IGN).

**La situacion.** La armadora *Pesquera del Este Simulada S.R.L.* recibio la notificacion de un sumario: su buque *DON FICTICIO* habria navegado **a menos de 6 nudos dentro de la Zona de Veda Permanente** (Res. CFP 26/2009). El limite oeste de esa veda es el **limite exterior del Mar Territorial**: 12 millas desde las lineas de base de la Ley 23.968. Hacia tierra de esa linea, las aguas son **provinciales (Chubut)** y la veda nacional no rige.

Ustedes son el **equipo tecnico** que contrata la armadora. Tienen el expediente (carpeta `expediente/`) y los datos (carpeta `recursos/`). La armadora quiere saber la verdad tecnica, **le convenga o no**.

### El expediente (todo simulado)

| # | Documento | Que trae |
|---|---|---|
| 01 | `01_notificacion_sumario_SIMULADA.pdf` | Hecho imputado, fecha, normas, sancion |
| 02 | `02_informe_monitoreo_autoridad_SIMULADO.pdf` | Las 5 posiciones que usa la autoridad (tambien en `recursos/posiciones_autoridad_SIMULADO.csv`) |
| 03 | `03_detalle_posiciones_proveedor_SIMULADO.html` | Export del VMS del proveedor de la armadora (tambien en `recursos/vms_don_ficticio_SIMULADO.csv`) |
| 04 | `04_parte_de_pesca_SIMULADO.pdf` | Declaracion jurada del capitan: horarios, lances, captura, zona |
| 05 | `05_croquis_capitan_SIMULADO.png` | Captura del ploter del capitan con *su* linea de 12 mn |
| 06 | `06_normativa_extractos.md` | Extractos de la normativa (real) |

Capas oficiales (reales, publicas): `recursos/espacios_maritimos_chubut.geojson` (limite del Mar Territorial, lineas de base, poligono) y `recursos/puntos_linea_base_ley23968_chubut_norte.csv`.

In [1]:
import os, json, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import shapely, pyproj
except ImportError:
    !pip -q install shapely pyproj
from shapely.geometry import shape, Point, LineString
from shapely.ops import transform
from pyproj import Transformer

RAW = ('https://raw.githubusercontent.com/PesquerosEnIA/curso-ia-produccion-pesquera/'
       'master/clase_04_datos_sensores/desafio_12mn/recursos/')

def ruta(nombre):
    local = os.path.join('..', 'recursos', nombre)
    if os.path.exists(local):
        return local
    destino = os.path.join('/tmp', nombre)
    if not os.path.exists(destino):
        urllib.request.urlretrieve(RAW + nombre, destino)
    return destino

# ── Herramientas que ya les damos hechas ────────────────────────────────────
def gm_a_decimal(txt):
    """'4320.483S' -> -43.34138 (grados + minutos decimales)"""
    num, hemis = txt[:-1], txt[-1]
    dec = int(num[:-6]) + float(num[-6:]) / 60
    return -dec if hemis in 'SW' else dec

capas = json.load(open(ruta('espacios_maritimos_chubut.geojson')))
geo = {f['properties']['capa']: shape(f['geometry']) for f in capas['features']}
MN = 1852.0
aeqd = Transformer.from_crs(4326, '+proj=aeqd +lat_0=-43.45 +lon_0=-64.85 +datum=WGS84', always_xy=True)
MT_m = transform(aeqd.transform, geo['mar_territorial'])
LIM_m = transform(aeqd.transform, geo['limite_mar_territorial'])

def dist_firmada(lon, lat):
    """metros a la linea de 12 mn: NEGATIVO = dentro (provincial), POSITIVO = fuera (nacional)"""
    p = Point(aeqd.transform(lon, lat))
    d = LIM_m.distance(p)
    return -d if MT_m.contains(p) else d

vms = pd.read_csv(ruta('vms_don_ficticio_SIMULADO.csv'))
vms['t_vms'] = pd.to_datetime(vms.fecha_hora_vms.str.split().str.join(' '), format='%b %d %Y %I:%M%p')
vms['lat'] = vms.lat_vms.apply(gm_a_decimal); vms['lon'] = vms.lon_vms.apply(gm_a_decimal)
vms = vms.sort_values('t_vms').reset_index(drop=True)

aut = pd.read_csv(ruta('posiciones_autoridad_SIMULADO.csv'))
aut['lat'] = aut.lat.apply(gm_a_decimal); aut['lon'] = aut.lon.apply(gm_a_decimal)
print(f'VMS: {len(vms)} posiciones | Autoridad: {len(aut)} posiciones | DATOS SIMULADOS')

VMS: 96 posiciones | Autoridad: 5 posiciones | DATOS SIMULADOS


---
## 1. Leer el expediente

Armen una tabla con lo que **afirma la autoridad**: fecha, horas, posiciones, norma, sancion, y que dice (o no dice) sobre la linea y la precision usadas. Pueden usar un LLM para extraerlo, pero **verifiquen cada dato contra el PDF**.

In [2]:
# TODO: tabla de hechos imputados (puede ser un DataFrame o un markdown)


---
## 2. ¿En que hora esta el VMS?

Comparen el primer y el ultimo movimiento del VMS con la zarpada (04:05) y el desembarque (19:25) del parte de pesca. ¿El VMS esta en hora local o en UTC? Creen una columna `t_local`.

In [3]:
# TODO: detectar el huso del VMS y crear vms['t_local']


---
## 3. ¿Las posiciones de la autoridad existen en el VMS?

El informe dice **22/03/2025**, "hora local". Busquen esas posiciones en el VMS. Si no aparecen, prueben desfases de hora y de fecha y encuentren el que hace coincidir lugares. ¿Que error cometio la autoridad?

In [4]:
# TODO: funcion que, para cada desfase en horas, mida la distancia entre la posicion de la autoridad y la del VMS


---
## 4. Dentro o fuera

Para cada posicion del VMS a menos de 6 nudos (en el mar) y para cada posicion de la autoridad, calculen `dist_firmada`. ¿Cuantas quedan fuera? ¿Cuanto tiempo? ¿A cuantos metros?

In [5]:
# TODO: distancias firmadas, conteos y un grafico distancia vs hora


---
## 5. Incertidumbre

¿La conclusion aguanta el error del GPS (σ 5–30 m) y el error grafico de una carta 1:275.000 (55–140 m)? Hagan un Monte Carlo o un analisis de margen. Distingan las posiciones **dudosas** de las **firmes**.

In [6]:
# TODO: Monte Carlo / margen por posicion


---
## 6. El croquis del capitan

La linea amarilla del ploter, ¿coincide con la oficial? Estimen cuantos metros esta corrida (pueden digitalizar dos o tres puntos de la imagen). ¿Que explica eso sobre la conducta del capitan?

In [7]:
# TODO: comparar la linea del ploter con la oficial


---
## 7. Coherencia con el parte

Lances, horarios, millas y zona declarada: ¿que coincide y que no con el VMS?

In [8]:
# TODO: lances detectados, millas recorridas, zona


---
## 8. Conclusiones tecnicas (maximo 1 pagina)

Redacten las conclusiones para la armadora: que esta probado, que es dudoso, que errores tiene el sumario y que conviene hacer (descargo, allanamiento parcial, pedido de prueba). **Una conclusion honesta puede no favorecer a quien la paga.**

In [9]:
# TODO: borrador de conclusiones (puede generarse a partir de los resultados)


---
## Entrega y rubrica

Entregar este notebook ejecutado + las conclusiones (punto 8).

| Criterio | Peso |
|---|---|
| Lectura correcta del expediente y de las normas | 15 % |
| Huso horario y desfase de la imputacion bien detectados | 20 % |
| Calculo dentro/fuera con la linea oficial y distancias en metros | 20 % |
| Tratamiento de la incertidumbre (que es firme y que es dudoso) | 20 % |
| Croquis y parte de pesca: coherencias e inconsistencias | 10 % |
| Conclusiones claras, honestas y utiles para decidir | 15 % |

> Recordatorio: **datos simulados**. No representan a ningun buque ni empresa real.